In [ ]:
import base64
import requests
import time
import pandas as pd
import os

# OpenAI API Key - Replace with your GPT API key
api_key = os.environ.get("OPENAI_API_KEY", "")  # set OPENAI_API_KEY

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("../../VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []


for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "../../Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Getting the base64 string
        base64_image = encode_image(image_path)
        
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }
        
        payload = {
            "model": "gpt-4.5-preview-2025-02-27", # Use GPT-4o for vision capabilities
            "max_tokens": 5000,
            "temperature": 0.0,
            "messages": [
                {
                    "role": "system",
                    "content": "You are an assistant, skilled in reading and interpreting visually represented data."
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": 'I am about to show you a graph and ask you a multiple-choice question about that graph. \n\n' +
                                   'Task 1: Data Extraction and Table Creation: First, explicitly list ALL numerical values you can identify on both axes, then create a structured table using markdown syntax that includes ALL data points you identified above with appropriate column headers with units. \n \n' +
                                   'Task 2: Sort the data: Sort the data in descending order by the numerical values. \n \n' +
                                   'Task 3: Data Verification and Error Handling: Double-check if your table matches ALL elements in the graph by comparing each value in your table with the graph and updating your table with correct values, verify the sorting is correct, and before proceeding, confirm all corrections have been made and use ONLY the corrected data for analysis. \n \n' +
                                   'Task 4: Question Analysis: Using ONLY the verified data in your table, compare EACH value individually with the reference value, for "less than" comparisons mark ALL values that are even slightly below the reference, for "greater than" comparisons mark ALL values that are even slightly above the reference, and show each comparison on a new line. \n \n' +
                                   'Provide your reasoning with specific references to table values. \n\n' +
                                   'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n' +
                                   'Let\'s solve this step by step.' +
                                   '\n\n' + question_text + " " + question_options
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        }
        
        # Add retry logic for overloaded errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                time_start = time.perf_counter()
                response_raw = requests.post(
                    "https://api.openai.com/v1/chat/completions",
                    headers=headers,
                    json=payload
                )
                time_end = time.perf_counter()
                
                response = response_raw.json()
                
                # If we get a rate limit error, wait and retry
                if 'error' in response and response['error'].get('code') == 'rate_limit_exceeded':
                    if retry < max_retries - 1:  # Don't sleep on the last retry
                        print(f"\nAPI rate limited, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                        time.sleep(retry_delay)
                        continue
                
                break  # If we get here, we either got a good response or a different error
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        print("\nAPI Response:", response)  # Debug print
        
        if 'error' in response:
            gpt_answer = f"Error: {response['error'].get('message', 'Unknown error')}"
            is_correct = "N/A"
        else:
            # Extract the answer after "Correct Answer: "
            full_response = response["choices"][0]["message"]["content"].strip()
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "./"
os.makedirs(results_dir, exist_ok=True)

results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "GPT_VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")